# Ligat ha'Al — top scorers & assists (Transfermarkt)

Scrapes **ligat-ha-al** on Transfermarkt (`ISR1`), `saison_id` = season start year.

- **Scorers:** `.../torschuetzenliste/.../saison_id/{YEAR}`
- **Assists:** `.../assistliste/.../saison_id/{YEAR}`

**CSV output:** `data/raw/player_stats/`  
**Columns:** `season`, `season_year`, `rank`, `player`, `player_url`, `position`, `nationality`, `age`, `club`, `club_url`, `matches`, `goals`, `assists`  
Set `SCORERS_TAIL_LAYOUT` in the scraper cell if needed (`matches_goals` vs `goals_assists`).


## 1. Environment setup


In [9]:
from pathlib import Path
from typing import Optional


def _find_root(start: Optional[Path] = None) -> Path:
    p = start or Path.cwd()
    for _ in range(6):
        if (p / "data").exists() or (p / ".git").exists() or (p / "notebooks").exists():
            return p
        p = p.parent
    return Path.cwd()


ROOT = _find_root()
DATA_DIR = ROOT / "data" / "raw"
PLAYER_STATS_DIR = DATA_DIR / "player_stats"
for d in (DATA_DIR, PLAYER_STATS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("PLAYER_STATS_DIR:", PLAYER_STATS_DIR)


ROOT: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks
PLAYER_STATS_DIR: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\raw\player_stats


## 2. HTTP helpers


In [10]:
import random
import time
from pathlib import Path
from typing import Optional

import pandas as pd
import requests

_USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 Version/17.0 Safari/605.1.15",
]

def _find_root() -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / "data").exists() or (p / ".git").exists() or (p / "notebooks").exists():
            return p
        p = p.parent
    return Path.cwd()

def ensure_environment():
    global ROOT, DATA_DIR, PLAYER_STATS_DIR
    ROOT = _find_root()
    DATA_DIR = ROOT / "data" / "raw"
    PLAYER_STATS_DIR = DATA_DIR / "player_stats"
    for d in (DATA_DIR, PLAYER_STATS_DIR, ROOT / "data" / "interim", ROOT / "data" / "processed"):
        d.mkdir(parents=True, exist_ok=True)
    return ROOT

def http_get(url: str, headers: Optional[dict] = None, retries: int = 3, timeout: int = 30) -> str:
    last_err = None
    sess = requests.Session()
    for attempt in range(1, retries + 1):
        ua = random.choice(_USER_AGENTS)
        hdrs = {"User-Agent": ua, "Accept-Language": "en-US,en;q=0.9"}
        if headers:
            hdrs.update(headers)
        try:
            resp = sess.get(url, headers=hdrs, timeout=timeout)
            resp.raise_for_status()
            return resp.text
        except Exception as e:
            last_err = e
            time.sleep(0.8 * attempt)
    raise last_err  # type: ignore

def save_csv(df: pd.DataFrame, path: Path, **kwargs):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding=kwargs.get("encoding", "utf-8-sig"))
    print("Saved:", path)

print("OK: http_get, save_csv")


OK: http_get, save_csv


## 3. Scraper (Transfermarkt `table.items`)


In [11]:
import re
from bs4 import BeautifulSoup

TM_BASE = "https://www.transfermarkt.com"
SCORERS_LIST_URL = TM_BASE + "/ligat-ha-al/torschuetzenliste/wettbewerb/ISR1/saison_id/{season}"
ASSISTS_URL = TM_BASE + "/ligat-ha-al/assistliste/wettbewerb/ISR1/saison_id/{season}"
SCORERS_TAIL_LAYOUT = "matches_goals"


def _norm_header(s: str) -> str:
    return re.sub(r"\s+", " ", s.replace("\xa0", " ").strip()).lower()


def _thead_labels(table) -> list[str]:
    thead = table.find("thead")
    if not thead:
        return []
    rows = thead.find_all("tr")
    best = []
    for tr in rows:
        labels = []
        for th in tr.find_all(["th", "td"]):
            labels.append(_norm_header(th.get_text(" ", strip=True)))
        if len(labels) > len(best):
            best = labels
    return best


def _cell_player_name(td) -> tuple[str, str]:
    a = td.find("a", class_="hauptlink") or td.find("a", href=re.compile(r"/profil/spieler/"))
    name = a.get_text(strip=True) if a else td.get_text(strip=True)
    href = (TM_BASE + a["href"]) if (a and a.get("href", "").startswith("/")) else (a.get("href") or "") if a else ""
    return name, href


def _player_cell_full(td) -> tuple[str, str, str]:
    position = ""
    inline = td.find("table", class_="inline-table")
    name, url = "", ""
    if inline:
        trs = inline.find_all("tr")
        if trs:
            a = trs[0].find("a", class_="hauptlink") or trs[0].find("a", href=re.compile(r"/profil/spieler/"))
            if a:
                name = a.get_text(strip=True)
                h = a.get("href", "")
                url = TM_BASE + h if h.startswith("/") else h
            if len(trs) > 1:
                position = trs[1].get_text(strip=True)
    if not name:
        name, url = _cell_player_name(td)
    return name, url, position


def _nat_cell(td) -> str:
    imgs = td.find_all("img", class_="flaggenrahmen")
    parts = []
    for img in imgs:
        t = (img.get("title") or img.get("alt") or "").strip()
        if t:
            parts.append(t)
    return "; ".join(parts)


def _cell_club_name(td) -> tuple[str, str]:
    a = td.find("a", class_="hauptlink") or td.find("a", href=re.compile(r"/startseite/verein/"))
    title = (a.get("title") or "").strip() if a else ""
    name = title or (a.get_text(strip=True) if a else td.get_text(strip=True))
    href = (TM_BASE + a["href"]) if (a and a.get("href", "").startswith("/")) else (a.get("href") or "") if a else ""
    return name, href


def _pick_column_index(headers: list[str], keywords: tuple[str, ...]) -> int | None:
    for i, h in enumerate(headers):
        for kw in keywords:
            if kw in h:
                return i
    return None


def _parse_int_text(t: str) -> int | None:
    t = t.strip().replace(".", "").replace(",", "")
    if not t or t == "-":
        return None
    m = re.search(r"-?\d+", t)
    return int(m.group(0)) if m else None


def _resolve_tail_indices(headers: list[str], n_cells: int, stat_mode: str) -> dict:
    hl = [_norm_header(h) for h in headers]
    while len(hl) < n_cells:
        hl.append("")
    matches = _pick_column_index(hl, ("match", "spiele", "spiel", "games"))
    goals = _pick_column_index(hl, ("goals", "tore"))
    assists = _pick_column_index(hl, ("assists", "vorlag"))

    if n_cells < 7:
        return {"matches": matches, "goals": goals, "assists": assists}

    if stat_mode == "assists":
        return {
            "matches": matches if matches is not None else 5,
            "goals": goals,
            "assists": assists if assists is not None else 6,
        }

    if goals is not None and assists is not None:
        return {"matches": matches, "goals": goals, "assists": assists}
    if matches is not None and goals is not None:
        return {"matches": matches, "goals": goals, "assists": assists}

    if SCORERS_TAIL_LAYOUT == "goals_assists":
        return {"matches": matches, "goals": 5, "assists": 6}
    return {"matches": matches if matches is not None else 5, "goals": goals if goals is not None else 6, "assists": assists}


def parse_items_table_page(html: str, stat_mode: str) -> pd.DataFrame:
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", class_="items")
    if not table:
        return pd.DataFrame()

    headers = _thead_labels(table)
    tbody = table.find("tbody")
    if not tbody:
        return pd.DataFrame()

    rows_out = []
    tail_ix = None
    for tr in tbody.find_all("tr", recursive=False):
        tds = tr.find_all("td", recursive=False)
        if len(tds) < 4:
            continue
        if tail_ix is None:
            tail_ix = _resolve_tail_indices(headers, len(tds), stat_mode)

        rank_t = tds[0].get_text(strip=True)
        if not rank_t.isdigit():
            continue

        player, player_url, position = _player_cell_full(tds[1]) if len(tds) > 1 else ("", "", "")
        if not player:
            continue

        nationality = _nat_cell(tds[2]) if len(tds) > 2 else ""
        age = _parse_int_text(tds[3].get_text(strip=True)) if len(tds) > 3 else None

        club, club_url = ("", "")
        if len(tds) > 4:
            club, club_url = _cell_club_name(tds[4])
            if not club.strip():
                club = tds[4].get_text(" ", strip=True)

        mi = tail_ix.get("matches")
        gi = tail_ix.get("goals")
        ai = tail_ix.get("assists")

        matches = _parse_int_text(tds[mi].get_text(strip=True)) if mi is not None and mi < len(tds) else None
        goals = _parse_int_text(tds[gi].get_text(strip=True)) if gi is not None and gi < len(tds) else None
        assists = _parse_int_text(tds[ai].get_text(strip=True)) if ai is not None and ai < len(tds) else None

        if stat_mode == "goals" and goals is None and len(tds) >= 1:
            goals = _parse_int_text(tds[-1].get_text(strip=True))
        if stat_mode == "goals" and matches is None and len(tds) >= 2:
            matches = _parse_int_text(tds[-2].get_text(strip=True))
        if stat_mode == "assists" and assists is None and len(tds) >= 1:
            assists = _parse_int_text(tds[-1].get_text(strip=True))
        if stat_mode == "assists" and matches is None and len(tds) >= 2:
            matches = _parse_int_text(tds[-2].get_text(strip=True))

        rows_out.append(
            {
                "rank": int(rank_t),
                "player": player,
                "player_url": player_url,
                "position": position,
                "nationality": nationality,
                "age": age,
                "club": club,
                "club_url": club_url,
                "matches": matches,
                "goals": goals,
                "assists": assists,
            }
        )

    if not rows_out:
        return pd.DataFrame()

    df = pd.DataFrame(rows_out)
    if stat_mode == "assists":
        df = df[
            [
                "rank",
                "player",
                "player_url",
                "position",
                "nationality",
                "age",
                "club",
                "club_url",
                "matches",
                "assists",
                "goals",
            ]
        ]
    else:
        df = df[
            [
                "rank",
                "player",
                "player_url",
                "position",
                "nationality",
                "age",
                "club",
                "club_url",
                "matches",
                "goals",
                "assists",
            ]
        ]
    return df


def fetch_all_pages(season_year: int, url_template: str, stat_mode: str, pause: float = 1.0) -> pd.DataFrame:
    base = url_template.format(season=season_year)
    all_parts: list[pd.DataFrame] = []
    page = 1
    while page <= 40:
        url = base if page == 1 else f"{base}/page/{page}"
        html = http_get(url)
        df = parse_items_table_page(html, stat_mode=stat_mode)
        if df.empty:
            break
        all_parts.append(df)
        if len(df) < 25:
            break
        page += 1
        time.sleep(pause)
    if not all_parts:
        return pd.DataFrame()
    return pd.concat(all_parts, ignore_index=True)


def scrape_top_scorers_season(season_year: int) -> pd.DataFrame | None:
    season_str = f"{season_year}/{str(season_year + 1)[-2:]}"
    print(f"Top scorers: {SCORERS_LIST_URL.format(season=season_year)}")
    try:
        df = fetch_all_pages(season_year, SCORERS_LIST_URL, "goals")
        if df.empty:
            print(f"  No rows for {season_str}")
            return None
        df.insert(0, "season", season_str)
        df.insert(1, "season_year", season_year)
        print(f"  OK: {len(df)} rows")
        return df
    except Exception as e:
        print(f"  Error {season_str}: {e}")
        return None


def scrape_top_assists_season(season_year: int) -> pd.DataFrame | None:
    season_str = f"{season_year}/{str(season_year + 1)[-2:]}"
    print(f"Top assists: {ASSISTS_URL.format(season=season_year)}")
    try:
        df = fetch_all_pages(season_year, ASSISTS_URL, "assists")
        if df.empty:
            print(f"  No rows for {season_str}")
            return None
        df.insert(0, "season", season_str)
        df.insert(1, "season_year", season_year)
        print(f"  OK: {len(df)} rows")
        return df
    except Exception as e:
        print(f"  Error {season_str}: {e}")
        return None


print("OK: scraper loaded")


OK: scraper loaded


## 4. Test one season


In [12]:
import time

ensure_environment()

test_year = 2024

df_s = scrape_top_scorers_season(test_year)
if df_s is not None:
    display(df_s.head(10))

time.sleep(1.2)

df_a = scrape_top_assists_season(test_year)
if df_a is not None:
    display(df_a.head(10))


Top scorers: https://www.transfermarkt.com/ligat-ha-al/torschuetzenliste/wettbewerb/ISR1/saison_id/2024
  OK: 171 rows


,season,season_year,rank,player,player_url,position,nationality,age,club,club_url,matches,goals,assists
0,2024/25,2024,1,Guy Melamed,https://www.transfermarkt.com/guy-melamed/prof...,Centre-Forward,Israel; Lithuania,32,for 2 clubs,,23,18,None
1,2024/25,2024,2,Dia Saba,https://www.transfermarkt.com/dia-saba/profil/...,Attacking Midfield,Israel,32,Maccabi Haifa,https://www.transfermarkt.com/maccabi-haifa/st...,25,15,None
2,2024/25,2024,3,Dean David,https://www.transfermarkt.com/dean-david/profi...,Centre-Forward,Israel,28,Maccabi Haifa,https://www.transfermarkt.com/maccabi-haifa/st...,21,13,None
3,2024/25,2024,4,Igor Zlatanovic,https://www.transfermarkt.com/igor-zlatanovic/...,Centre-Forward,Serbia,27,Maccabi Netanya,https://www.transfermarkt.com/maccabi-netanya/...,22,13,None
4,2024/25,2024,5,Dor Turgeman,https://www.transfermarkt.com/dor-turgeman/pro...,Centre-Forward,Israel,21,Maccabi Tel Aviv,https://www.transfermarkt.com/maccabi-tel-aviv...,22,13,None
5,2024/25,2024,6,Kings Kangwa,https://www.transfermarkt.com/kings-kangwa/pro...,Central Midfield,Zambia,25,Hapoel Beer Sheva,https://www.transfermarkt.com/hapoel-beer-shev...,25,10,None
6,2024/25,2024,7,Alon Turgeman,https://www.transfermarkt.com/alon-turgeman/pr...,Centre-Forward,Israel,33,Hapoel Beer Sheva,https://www.transfermarkt.com/hapoel-beer-shev...,24,10,None
7,2024/25,2024,8,Lior Refaelov,https://www.transfermarkt.com/lior-refaelov/pr...,Attacking Midfield,Israel; Belgium,38,Maccabi Haifa,https://www.transfermarkt.com/maccabi-haifa/st...,23,9,None
8,2024/25,2024,9,Hamode Kanaan,https://www.transfermarkt.com/hamode-kanaan/pr...,Attacking Midfield,Israel,25,FC Ashdod,https://www.transfermarkt.com/fc-ashdod/starts...,19,8,None
9,2024/25,2024,10,Dor Peretz,https://www.transfermarkt.com/dor-peretz/profi...,Central Midfield,Israel; Portugal,29,Maccabi Tel Aviv,https://www.transfermarkt.com/maccabi-tel-aviv...,26,7,None


Top assists: https://www.transfermarkt.com/ligat-ha-al/assistliste/wettbewerb/ISR1/saison_id/2024
  OK: 151 rows


,season,season_year,rank,player,player_url,position,nationality,age,club,club_url,matches,assists,goals
0,2024/25,2024,1,Dia Saba,https://www.transfermarkt.com/dia-saba/profil/...,Attacking Midfield,Israel,32,Maccabi Haifa,https://www.transfermarkt.com/maccabi-haifa/st...,25,11,None
1,2024/25,2024,2,Dor Micha,https://www.transfermarkt.com/dor-micha/profil...,Attacking Midfield,Israel; Portugal,33,Beitar Jerusalem,https://www.transfermarkt.com/beitar-jerusalem...,25,10,None
2,2024/25,2024,3,Hamode Kanaan,https://www.transfermarkt.com/hamode-kanaan/pr...,Attacking Midfield,Israel,25,FC Ashdod,https://www.transfermarkt.com/fc-ashdod/starts...,19,8,None
3,2024/25,2024,4,Freddy Vargas,https://www.transfermarkt.com/freddy-vargas/pr...,Left Winger,Venezuela,25,Maccabi Netanya,https://www.transfermarkt.com/maccabi-netanya/...,22,8,None
4,2024/25,2024,5,Yarden Shua,https://www.transfermarkt.com/yarden-shua/prof...,Centre-Forward,Israel,25,Beitar Jerusalem,https://www.transfermarkt.com/beitar-jerusalem...,23,7,None
5,2024/25,2024,6,Dolev Haziza,https://www.transfermarkt.com/dolev-haziza/pro...,Left Winger,Israel; France,29,Maccabi Haifa,https://www.transfermarkt.com/maccabi-haifa/st...,25,7,None
6,2024/25,2024,7,Kings Kangwa,https://www.transfermarkt.com/kings-kangwa/pro...,Central Midfield,Zambia,25,Hapoel Beer Sheva,https://www.transfermarkt.com/hapoel-beer-shev...,25,7,None
7,2024/25,2024,8,Maxim Plakushchenko,https://www.transfermarkt.com/maxim-plakushche...,Central Midfield,Israel; Ukraine,29,Maccabi Netanya,https://www.transfermarkt.com/maccabi-netanya/...,23,6,None
8,2024/25,2024,9,Dan Biton,https://www.transfermarkt.com/dan-biton/profil...,Right Winger,Israel,29,Hapoel Beer Sheva,https://www.transfermarkt.com/hapoel-beer-shev...,23,6,None
9,2024/25,2024,10,Gabi Kanichowsky,https://www.transfermarkt.com/gabi-kanichowsky...,Attacking Midfield,Israel,27,Maccabi Tel Aviv,https://www.transfermarkt.com/maccabi-tel-aviv...,24,6,None


## 5. Multi-season collection (default 2006–2025)


In [13]:
import time

ensure_environment()

start_year = 2006
end_year = 2025
seasons = list(range(start_year, end_year + 1))

print(f"Seasons: {len(seasons)} ({start_year}–{end_year})")
print("=" * 80)

all_scorers = []
all_assists = []
failed_s = []
failed_a = []

for season_year in seasons:
    season_str = f"{season_year}/{str(season_year + 1)[-2:]}"
    print(f"\n[{season_str}]")

    path_s = PLAYER_STATS_DIR / f"top_scorers_{season_year}_{str(season_year + 1)[-2:]}_ligat_haal_transfermarkt.csv"
    if path_s.exists():
        try:
            all_scorers.append(pd.read_csv(path_s))
            print("  scorers: exists", path_s.name)
        except Exception as e:
            print("  scorers: re-read failed", e)
            d = scrape_top_scorers_season(season_year)
            if d is not None:
                save_csv(d, path_s)
                all_scorers.append(d)
            else:
                failed_s.append(season_str)
    else:
        d = scrape_top_scorers_season(season_year)
        if d is not None:
            save_csv(d, path_s)
            all_scorers.append(d)
        else:
            failed_s.append(season_str)
        time.sleep(1.2)

    path_a = PLAYER_STATS_DIR / f"top_assists_{season_year}_{str(season_year + 1)[-2:]}_ligat_haal_transfermarkt.csv"
    if path_a.exists():
        try:
            all_assists.append(pd.read_csv(path_a))
            print("  assists: exists", path_a.name)
        except Exception as e:
            print("  assists: re-read failed", e)
            d = scrape_top_assists_season(season_year)
            if d is not None:
                save_csv(d, path_a)
                all_assists.append(d)
            else:
                failed_a.append(season_str)
    else:
        d = scrape_top_assists_season(season_year)
        if d is not None:
            save_csv(d, path_a)
            all_assists.append(d)
        else:
            failed_a.append(season_str)
        time.sleep(1.2)

print("\n" + "=" * 80)
print("scorers OK:", len(all_scorers), "failed:", failed_s)
print("assists OK:", len(all_assists), "failed:", failed_a)

if all_scorers:
    comb = pd.concat(all_scorers, ignore_index=True)
    out = PLAYER_STATS_DIR / "top_scorers_all_seasons_ligat_haal_transfermarkt.csv"
    save_csv(comb, out)
    display(comb.groupby("season").size().to_frame("players"))

if all_assists:
    comb = pd.concat(all_assists, ignore_index=True)
    out = PLAYER_STATS_DIR / "top_assists_all_seasons_ligat_haal_transfermarkt.csv"
    save_csv(comb, out)
    display(comb.groupby("season").size().to_frame("players"))


Seasons: 20 (2006–2025)

[2006/07]
Top scorers: https://www.transfermarkt.com/ligat-ha-al/torschuetzenliste/wettbewerb/ISR1/saison_id/2006
  OK: 144 rows
Saved: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\raw\player_stats\top_scorers_2006_07_ligat_haal_transfermarkt.csv
Top assists: https://www.transfermarkt.com/ligat-ha-al/assistliste/wettbewerb/ISR1/saison_id/2006
  OK: 94 rows
Saved: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\raw\player_stats\top_assists_2006_07_ligat_haal_transfermarkt.csv

[2007/08]
Top scorers: https://www.transfermarkt.com/ligat-ha-al/torschuetzenliste/wettbewerb/ISR1/saison_id/2007
  OK: 135 rows
Saved: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\raw\player_stats\top_scorers_2007_08_ligat_haal_transfermarkt.csv
Top assists: https://www.transfermarkt.com/ligat-ha-al/assistliste/wettbewerb/ISR1/saison_id/2007
  OK: 5 rows
Saved: c:\Users\nitib\dev-lab\ligat_haal_proje

,players
season,
2006/07,144
2007/08,135
2008/09,138
2009/10,176
2010/11,171
2011/12,177
2012/13,161
2013/14,147
2014/15,158


Saved: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\raw\player_stats\top_assists_all_seasons_ligat_haal_transfermarkt.csv


,players
season,
2006/07,94
2007/08,5
2008/09,5
2009/10,1000
2010/11,60
2011/12,166
2012/13,135
2013/14,135
2014/15,123


איסוף טבלת מלכי השערים לעונה הנוכחית 

In [ ]:
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from bs4 import BeautifulSoup


BASE_URL = "https://www.transfermarkt.com/ligat-haal/torschuetzenliste/wettbewerb/ISR1/saison_id/2025"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


def clean_text(value: str) -> str:
    return " ".join(value.replace("\xa0", " ").split())


def parse_int(value: str):
    value = clean_text(value)
    if value in ["", "-", "—"]:
        return None
    return int(value.replace(",", ""))


def extract_img_title(td):
    img = td.find("img")
    if not img:
        return None
    return img.get("title") or img.get("alt")


def scrape_page(page: int) -> list[dict]:
    if page == 1:
        url = BASE_URL
    else:
        url = f"{BASE_URL}/page/{page}"

    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")

    table = soup.select_one("table.items")
    if table is None:
        return []

    rows = table.select("tbody > tr.odd, tbody > tr.even")
    results = []

    for row in rows:
        # חשוב: recursive=False כדי לא לקחת תאים מתוך הטבלה הפנימית של השחקן
        cols = row.find_all("td", recursive=False)

        if len(cols) < 7:
            continue

        rank = clean_text(cols[0].get_text())

        player_link = cols[1].select_one("table.inline-table td.hauptlink a")
        player_name = clean_text(player_link.get_text()) if player_link else None
        player_url = (
            "https://www.transfermarkt.com" + player_link.get("href")
            if player_link and player_link.get("href")
            else None
        )

        position_cell = cols[1].select_one("table.inline-table tr:nth-of-type(2) td")
        position = clean_text(position_cell.get_text()) if position_cell else None

        nationalities = []
        for img in cols[2].find_all("img"):
            title = img.get("title") or img.get("alt")
            if title:
                nationalities.append(title)

        age = clean_text(cols[3].get_text())
        club = extract_img_title(cols[4])

        appearances = clean_text(cols[5].get_text())
        goals = clean_text(cols[6].get_text())

        results.append(
            {
                "season": "2025/26",
                "rank": parse_int(rank),
                "player": player_name,
                "position": position,
                "nationality": ", ".join(nationalities),
                "age": parse_int(age),
                "club": club,
                "appearances": parse_int(appearances),
                "goals": parse_int(goals),
                "player_url": player_url,
                "source": url,
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            }
        )

    return results


def scrape_all_pages(max_pages: int = 10) -> pd.DataFrame:
    all_rows = []

    for page in range(1, max_pages + 1):
        print(f"Scraping page {page}...")
        rows = scrape_page(page)

        if not rows:
            print(f"No rows found on page {page}. Stopping.")
            break

        all_rows.extend(rows)
        time.sleep(1)

    df = pd.DataFrame(all_rows)

    if not df.empty:
        df = df.drop_duplicates(subset=["player", "club", "season"])
        df = df.sort_values(["goals", "appearances"], ascending=[False, True])

    return df


if __name__ == "__main__":
    df = scrape_all_pages(max_pages=10)

    print(df.head(20))
    print(f"\nTotal rows: {len(df)}")

    df.to_csv("ligat_haal_top_scorers_2025_26.csv", index=False, encoding="utf-8-sig")
    df.to_excel("ligat_haal_top_scorers_2025_26.xlsx", index=False)

    print("\nSaved:")
    print("ligat_haal_top_scorers_2025_26.csv")
    print("ligat_haal_top_scorers_2025_26.xlsx")

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
     season  rank            player            position  \
0   2025/26     1         Dan Biton        Right Winger   
1   2025/26     2       Stav Turiel        Right Winger   
3   2025/26     4       Yarden Shua      Centre-Forward   
2   2025/26     3   Adrián Ugarriza      Centre-Forward   
4   2025/26     5        Dor Peretz    Central Midfield   
5   2025/26     6       Omer Atzili        Right Winger   
6   2025/26     7      Kings Kangwa    Central Midfield   
7   2025/26     8        Ido Shahar    Central Midfield   
9   2025/26    10       Guy Melamed      Centre-Forward   
10  2025/26    11  Trivante Stewart      Centre-Forward   
8   2025/26     9        Javon East        Right Winger   
17  2025/26    18       Márk Koszta      Centre-Forward   
12  2025/26    13      Matheus Davó      C